In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import gc
import pickle
import time

import sys
from pathlib import Path

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

from models import *
from models.kdv.trainer import *
from models.kdv.visualizer import *

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
testing_seeds = [42, 72, 83, 27, 81, 174, 9705, 2006, 172, 1969, 1975, 92625, 10000, 2342345, 986689]

INIT_PARAMS = dict(
    num_solitons             = 1,
    n_hidden_layers          = 4, 
    n_neurons_per_layer      = 32, 
    activation               = nn.Tanh,
    seed                     = 72, 
    verbose                  = False,
)

soliton_params = KDV(INIT_PARAMS).soliton_params

In [ ]:

CONS_DOMAIN_PARAMS = dict(
    'n_collocation': 50000,
    'n_initial': 30000,
    'n_boundary': 10000,
    'n_momentum': 0,
    'n_energy': 0,
    'n_hamiltonian': 15,
    'soliton_params': soliton_params
)

CONS_FREE_DOMAIN_PARAMS = dict(
    'n_collocation': 50000,
    'n_initial': 30000,
    'n_boundary': 10000,
    'n_momentum': 0,
    'n_energy': 0,
    'n_hamiltonian': 0,
    'soliton_params': soliton_params
)

TRAIN_WEIGHTS_CONS = dict( #seperated out from the train params
    w_ic                     = 5.0,    
    w_bc                     = 2.0,    
    w_pde                    = 15.0,
    w_momentum               = 0.0,
    w_energy                 = 0.0,
    w_hamiltonian            = 0.5
)

TRAIN_WEIGHTS = dict( #seperated out from the train params
    w_ic                     = 5.0,    
    w_bc                     = 2.0,    
    w_pde                    = 15.0,
    w_momentum               = 0.0,
    w_energy                 = 0.0,
    w_hamiltonian            = 0.0
)

domain_cons = setup_training_domain(**CONS_DOMAIN_PARAMS)
domain_no_cons = setup_training_domain(**CONS_FREE_DOMAIN_PARAMS)

results = {
    'time': [],
    'mae': [],
    'max_error': []
}

for seed in testing_seeds:

    INIT_PARAMS = dict(
    num_solitons             = 1,
    n_hidden_layers          = 4, 
    n_neurons_per_layer      = 32, 
    activation               = nn.Tanh,
    seed                     = seed, 
    verbose                  = False,
    )

    model = KDV(INIT_PARAMS)
    print(f'Starting SEED {seed}')
    start_time = time.time()
    adam(model.neural_net, domain_no_cons, TRAIN_WEIGHTS, 1000, 1e-3)
    lbfgs_test(model.neural_net, domain_cons, TRAIN_WEIGHTS_CONS, 50000)
    error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')
    results['time'].append(time.time() - start_time)
    results['mae'].append(error_stats.mae)
    results['max_error'].append(error_stats.max_error)

    print(f'Seed {seed} completed')
    del model
    gc.collect
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'CONSERVATION LBFGS ONLY')
print(f'AVG TIME: {np.mean(results['time'])} | STD TIME: {np.std(results['time'])}')
print(f'MAE: {np.mean(results['mae'])} | STD MAE: {np.std(results['mae'])}')
print(f'AVG MAX ERR {np.mean(results["max_error"])} | STD MAX ERR {np.std(results["max_error"])}')

In [ ]:

CONS_DOMAIN_PARAMS = dict(
    'n_collocation': 50000,
    'n_initial': 30000,
    'n_boundary': 10000,
    'n_momentum': 0,
    'n_energy': 0,
    'n_hamiltonian': 15,
    'soliton_params': soliton_params
)

CONS_FREE_DOMAIN_PARAMS = dict(
    'n_collocation': 50000,
    'n_initial': 30000,
    'n_boundary': 10000,
    'n_momentum': 0,
    'n_energy': 0,
    'n_hamiltonian': 0,
    'soliton_params': soliton_params
)

TRAIN_WEIGHTS_CONS = dict( #seperated out from the train params
    w_ic                     = 5.0,    
    w_bc                     = 2.0,    
    w_pde                    = 15.0,
    w_momentum               = 0.0,
    w_energy                 = 0.0,
    w_hamiltonian            = 0.5
)

TRAIN_WEIGHTS = dict( #seperated out from the train params
    w_ic                     = 5.0,    
    w_bc                     = 2.0,    
    w_pde                    = 15.0,
    w_momentum               = 0.0,
    w_energy                 = 0.0,
    w_hamiltonian            = 0.0
)

domain_cons = setup_training_domain(**CONS_DOMAIN_PARAMS)
domain_no_cons = setup_training_domain(**CONS_FREE_DOMAIN_PARAMS)

results = {
    'time': [],
    'mae': [],
    'max_error': []
}

for seed in testing_seeds:

    INIT_PARAMS = dict(
    num_solitons             = 1,
    n_hidden_layers          = 4, 
    n_neurons_per_layer      = 32, 
    activation               = nn.Tanh,
    seed                     = seed, 
    verbose                  = False,
    )

    model = KDV(INIT_PARAMS)
    print(f'Starting SEED {seed}')
    start_time = time.time()
    adam(model.neural_net, domain_no_cons, TRAIN_WEIGHTS, 1000, 1e-3)
    lbfgs_test(model.neural_net, domain_no_cons, TRAIN_WEIGHTS, 50000)
    lbfgs_test(model.neural_net, domain_cons, TRAIN_WEIGHTS_CONS, 50000)
    error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')
    results['time'].append(time.time() - start_time)
    results['mae'].append(error_stats.mae)
    results['max_error'].append(error_stats.max_error)

    print(f'Seed {seed} completed')
    del model
    gc.collect
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'2 PHASE LBFGS ONLY')
print(f'AVG TIME: {np.mean(results['time'])} | STD TIME: {np.std(results['time'])}')
print(f'MAE: {np.mean(results['mae'])} | STD MAE: {np.std(results['mae'])}')
print(f'AVG MAX ERR {np.mean(results["max_error"])} | STD MAX ERR {np.std(results["max_error"])}')